# 🌱 AgriDoc AI - Huấn luyện Mô hình Nhận diện Bệnh Cây trồng

Notebook này huấn luyện mô hình CNN (Convolutional Neural Network) để nhận diện bệnh trên lá cây.

**Hướng dẫn sử dụng:**
1. Dán đoạn code tải dữ liệu từ Roboflow vào Ô 1 bên dưới.
2. Bật GPU: Settings → Accelerator → GPU T4.
3. Bấm Run All và chờ khoảng 15-30 phút.
4. Tải 2 file `best_model.pt` và `class_indices.json` từ Output về máy.

## Ô 1: Tải dữ liệu từ Roboflow và Gom vào thư mục Dataset
**👉 Dán đoạn code Python từ Roboflow vào ô code bên dưới (thay thế dòng chú thích).**

In [ ]:
# ====================================================================
# 👇 DÁN ĐOẠN CODE TỪ ROBOFLOW VÀO ĐÂY (thay thế toàn bộ ô này) 👇
# ====================================================================
# Ví dụ:
# !pip install roboflow
# from roboflow import Roboflow
# rf = Roboflow(api_key="YOUR_API_KEY")
# project = rf.workspace("your-workspace").project("your-project")
# version = project.version(1)
# dataset = version.download("folder")
# ====================================================================

In [ ]:
import os
import shutil

# rf_path chính là biến dataset.location do Roboflow tạo ra ở trên
rf_path = dataset.location
os.makedirs("Dataset", exist_ok=True)

# Gom tất cả ảnh từ train, valid, test vào chung 1 thư mục Dataset
for split_name in ['train', 'valid', 'test']:
    split_dir = os.path.join(rf_path, split_name)
    if os.path.exists(split_dir):
        for class_name in os.listdir(split_dir):
            src_class = os.path.join(split_dir, class_name)
            if os.path.isdir(src_class):  # Skip README files from Roboflow
                dst_class = os.path.join("Dataset", class_name)
                os.makedirs(dst_class, exist_ok=True)
                for img in os.listdir(src_class):
                    shutil.copy(os.path.join(src_class, img), os.path.join(dst_class, img))

# Verify the dataset
classes = [d for d in os.listdir("Dataset") if os.path.isdir(os.path.join("Dataset", d))]
total_images = sum(len(os.listdir(os.path.join("Dataset", c))) for c in classes)
print(f"✅ Gom dữ liệu thành công!")
print(f"📁 Số loại bệnh: {len(classes)}")
print(f"🖼️ Tổng số ảnh: {total_images}")
print(f"📋 Danh sách: {classes}")

## Ô 2: Import thư viện và thiết lập GPU

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import json
from datetime import datetime

import torch
import torch.nn as nn
from torchvision import datasets, transforms
from torch.utils.data.sampler import SubsetRandomSampler

# Auto-detect GPU or CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🖥️ Sử dụng thiết bị: {device}")
if device.type == 'cuda':
    print(f"🚀 GPU: {torch.cuda.get_device_name(0)}")
else:
    print("⚠️ Đang chạy trên CPU. Hãy bật GPU trong Settings → Accelerator → GPU T4 để nhanh hơn!")

## Ô 3: Load và chia dữ liệu (85% Train, 15% Test)

In [ ]:
import torchvision.transforms as transforms
from torchvision import datasets
from torch.utils.data import DataLoader
from torch.utils.data.sampler import SubsetRandomSampler
import numpy as np

# Vì dữ liệu đã được Augment bên Roboflow, ta chỉ cần Resize và Normalize cơ bản
transform = transforms.Compose([
    transforms.Resize(255),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

dataset_path = "Dataset"
dataset = datasets.ImageFolder(dataset_path, transform=transform)

# Chia dữ liệu 85% Train, 15% Validation
num_data = len(dataset)
indices = list(range(num_data))
np.random.shuffle(indices)

split_train = int(np.floor(0.85 * num_data))
train_idx, val_idx = indices[:split_train], indices[split_train:]

train_sampler = SubsetRandomSampler(train_idx)
validation_sampler = SubsetRandomSampler(val_idx)

train_loader = DataLoader(dataset, batch_size=32, sampler=train_sampler)
validation_loader = DataLoader(dataset, batch_size=32, sampler=validation_sampler)
test_loader = validation_loader # Dùng chung val làm test cho đơn giản

print("✅ Đã chuẩn bị xong Dữ liệu và DataLoader!")
print(f"Tổng số ảnh: {num_data} (Train: {len(train_idx)}, Validation: {len(val_idx)})")
print(f"Số loại bệnh: {len(dataset.classes)}")



## Ô 4: Xây dựng mạng CNN (Convolutional Neural Network)

In [ ]:
from torchvision import models

# Sử dụng Transfer Learning với ResNet18 thay vì tự viết CNN
model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)

# Thay thế lớp phân loại cuối cùng (fc) cho phù hợp với số loại bệnh của chúng ta
num_ftrs = model.fc.in_features
model.fc = nn.Linear(num_ftrs, len(train_loader.dataset.classes))

# Đưa mô hình lên GPU
model = model.to(device)

print(f"✅ Mô hình ResNet18 đã sẵn sàng với {len(train_loader.dataset.classes)} loại bệnh!")
print(f"📐 Tổng tham số: {sum(p.numel() for p in model.parameters()):,}")


## Ô 5: Huấn luyện mô hình (Training)

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.0001) # Learning rate nhỏ cho Transfer Learning

def batch_gd(model, criterion, train_loader, validation_loader, epochs):
    train_losses = np.zeros(epochs)
    validation_losses = np.zeros(epochs)

    for e in range(epochs):
        t0 = datetime.now()
        model.train()
        train_loss = []
        for inputs, targets in train_loader:
            inputs, targets = inputs.to(device), targets.to(device)
            optimizer.zero_grad()
            output = model(inputs)
            loss = criterion(output, targets)
            train_loss.append(loss.item())
            loss.backward()
            optimizer.step()

        train_loss = np.mean(train_loss)

        model.eval()
        validation_loss = []
        with torch.no_grad():
            for inputs, targets in validation_loader:
                inputs, targets = inputs.to(device), targets.to(device)
                output = model(inputs)
                loss = criterion(output, targets)
                validation_loss.append(loss.item())

        validation_loss = np.mean(validation_loss)
        train_losses[e] = train_loss
        validation_losses[e] = validation_loss
        dt = datetime.now() - t0

        print(f"Epoch {e+1}/{epochs} | Train Loss: {train_loss:.3f} | Val Loss: {validation_loss:.3f} | Thời gian: {dt}")

    return train_losses, validation_losses

# === BẮT ĐẦU HUẤN LUYỆN ===
print("🚀 Bắt đầu huấn luyện...")
train_losses, validation_losses = batch_gd(
    model, criterion, train_loader, validation_loader, 500
)


## Ô 6: Đánh giá độ chính xác (Accuracy)

In [ ]:
def accuracy(loader):
    model.eval()
    n_correct = 0
    n_total = 0
    with torch.no_grad():
        for inputs, targets in loader:
            inputs, targets = inputs.to(device), targets.to(device)
            outputs = model(inputs)
            _, predictions = torch.max(outputs, 1)
            n_correct += (predictions == targets).sum().item()
            n_total += targets.shape[0]
    return n_correct / n_total

train_acc = accuracy(train_loader)
test_acc = accuracy(test_loader)
val_acc = accuracy(validation_loader)

print(f"📊 Kết quả đánh giá:")
print(f"   🔹 Train Accuracy:      {train_acc*100:.1f}%")
print(f"   🔹 Validation Accuracy: {val_acc*100:.1f}%")
print(f"   🔹 Test Accuracy:       {test_acc*100:.1f}%")

## Ô 7: Biểu đồ Loss (Train vs Validation)

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(train_losses, label='Train Loss', linewidth=2)
plt.plot(validation_losses, label='Validation Loss', linewidth=2)
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('AgriDoc AI - Training Progress')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## Ô 8: 💾 LƯU MÔ HÌNH (Quan trọng nhất!)
Sau khi chạy xong ô này, bạn sẽ thấy 2 file xuất hiện ở mục **Output** bên phải:
- `best_model.pt` → Trọng số AI
- `class_indices.json` → Danh sách tên bệnh

**Tải cả 2 file về và chép vào thư mục `backend/saved_models/` là xong!**

In [ ]:
# === LƯU TRỌNG SỐ MÔ HÌNH ===
torch.save(model.state_dict(), 'best_model.pt')
print("✅ Đã lưu: best_model.pt")

# === LƯU DANH SÁCH TÊN BỆNH ===
class_indices = {str(v): k for k, v in dataset.class_to_idx.items()}
with open('class_indices.json', 'w', encoding='utf-8') as f:
    json.dump(class_indices, f, ensure_ascii=False, indent=4)
print("✅ Đã lưu: class_indices.json")

# === VERIFY ===
print(f"\n🎉 HOÀN TẤT! Đã xuất 2 file:")
print(f"   📦 best_model.pt ({os.path.getsize('best_model.pt') / 1024 / 1024:.1f} MB)")
print(f"   📋 class_indices.json ({len(class_indices)} loại bệnh)")
print(f"\n👉 Hãy tải 2 file này từ Output (bên phải) về máy tính.")
print(f"👉 Chép đè vào thư mục: backend/saved_models/")
print(f"👉 Khởi động lại Backend là xong!")